In [44]:
import pandas as pd
import networkx as nx
import importlib

# utils imports
from utils.geometry_utils import rebuild_polygons
import utils.graph_utils as graph_utils

# reload in case graph_utils changed
importlib.reload(graph_utils)
from utils.graph_utils import build_graph

In [45]:
rooms_df = pd.read_csv("rooms_dataset.csv")

rooms_df.head()

,room_type,area,centroid_x,centroid_y,num_vertices,points,plan_id
0,Outdoor Terrace,248496.0296,1260.503333,387.083333,6,"[(1030.81, 507.29), (1030.81, 146.67), (1719.8...",6044
1,Entry Lobby,54825.5028,1414.466667,1211.093333,9,"[(1469.34, 1137.48), (1469.34, 1148.48), (1530...",6044
2,Kitchen,67064.8976,1576.103333,1320.606667,6,"[(1712.49, 1478.05), (1474.05, 1478.05), (1474...",6044
3,LivingRoom,221476.7801,1482.916667,934.763333,6,"[(1712.49, 1137.48), (1469.34, 1137.48), (1334...",6044
4,Utility Laundry,20826.7515,1121.873333,1377.933333,6,"[(1190.03, 1478.05), (1051.17, 1478.05), (1051...",6044


In [46]:
rooms_df = rebuild_polygons(rooms_df)

In [48]:
def compute_spatial_metrics(G):

    graph_density = nx.density(G)

    components = list(nx.connected_components(G))

    path_lengths = []
    mean_depths = []

    for comp in components:

        subgraph = G.subgraph(comp)

        if len(subgraph.nodes) > 1:

            avg_shortest = nx.average_shortest_path_length(subgraph)
            path_lengths.append(avg_shortest)

            lengths = dict(nx.shortest_path_length(subgraph))

            depths = []

            for node in lengths:
                depths.append(sum(lengths[node].values())/(len(lengths[node])-1))

            mean_depths.append(sum(depths)/len(depths))

    if len(path_lengths) > 0:
        avg_shortest_path = sum(path_lengths)/len(path_lengths)
        mean_depth = sum(mean_depths)/len(mean_depths)
    else:
        avg_shortest_path = 0
        mean_depth = 0

    if mean_depth != 0:
        integration = 1/mean_depth
    else:
        integration = 0

    return {
        "graph_density": graph_density,
        "avg_shortest_path": avg_shortest_path,
        "mean_depth": mean_depth,
        "integration": integration
    }
    

In [49]:
results = []

for plan_id in rooms_df["plan_id"].unique():

    plan_rooms = rooms_df[rooms_df["plan_id"] == plan_id]

    G = build_graph(plan_rooms)

    metrics = compute_spatial_metrics(G)

    metrics["plan_id"] = plan_id

    results.append(metrics)

In [50]:
spatial_features_df = pd.DataFrame(results)

spatial_features_df.head()

,graph_density,avg_shortest_path,mean_depth,integration,plan_id
0,0.066667,1.166667,1.166667,0.857143,6044
1,0.000000,0.000000,0.000000,0.000000,2564
2,0.200000,1.666667,1.666667,0.600000,6165
3,0.014706,1.000000,1.000000,1.000000,1021
4,0.016667,1.000000,1.000000,1.000000,5507


In [51]:
spatial_features_df.to_csv("spatial_syntax_features.csv", index=False)